# AeroDataBox API (via RapidAPI)

`aerodatabox.p.rapidapi.com`. This notebook calls it and records what it returns.


In [3]:
#import libraries
import json
import os
import time
import requests
from dotenv import load_dotenv
from datetime import date, timedelta
import pandas as pd


In [4]:
load_dotenv(override=True)
RAPIDAPI_KEY = os.environ["RAPIDAPI_KEY"].strip()

HOST = "aerodatabox.p.rapidapi.com"
HEADERS = {
    "x-rapidapi-host": HOST,
    "x-rapidapi-key": RAPIDAPI_KEY,
}


def fetch(path: str, params: dict | None = None):
    url = f"https://{HOST}{path}"
    r = requests.get(url, headers=HEADERS, params=params or {}, timeout=30)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

## 1. Flight by number and date

Path: `/flights/number/{code}/{date}`. `dateLocalRole=Both` is required.

In [5]:
status, payload = fetch(
    "/flights/number/AS2223/2026-08-12",
    {
        "withAircraftImage": "false",
        "withLocation": "false",
        "withFlightPlan": "false",
        "dateLocalRole": "Both",
    },
)
print(f"status {status}")
print(json.dumps(payload, indent=2)[:3000])

status 200
[
  {
    "greatCircleDistance": {
      "meter": 1324245.9,
      "km": 1324.25,
      "mile": 822.85,
      "nm": 715.04,
      "feet": 4344638.77
    },
    "departure": {
      "airport": {
        "icao": "KSAN",
        "iata": "SAN",
        "name": "San Diego",
        "shortName": "San Diego",
        "municipalityName": "San Diego",
        "location": {
          "lat": 32.7336,
          "lon": -117.19
        },
        "countryCode": "US",
        "timeZone": "America/Los_Angeles"
      },
      "scheduledTime": {
        "utc": "2026-08-13 02:48Z",
        "local": "2026-08-12 19:48-07:00"
      },
      "terminal": "2",
      "quality": [
        "Basic"
      ]
    },
    "arrival": {
      "airport": {
        "icao": "KRDM",
        "iata": "RDM",
        "name": "Redmond Roberts Field",
        "shortName": "Roberts Field",
        "municipalityName": "Redmond",
        "location": {
          "lat": 44.2541,
          "lon": -121.15
        },
        "c

## 2. How far ahead the schedule is published

204 means no content: the request succeeded, but no flight exists for that code on
that date - either because it does not operate that day, or because the schedule for
it has not been published yet.

In [6]:

today = date.today()
rows = []
for offset in [1, 3, 5, 7, 10, 14, 21, 30, 45, 60]:
    target = (today + timedelta(days=offset)).isoformat()
    status, payload = fetch(
        f"/flights/number/AS2223/{target}", {"dateLocalRole": "Both"}
    )
    has_data = isinstance(payload, list) and len(payload) > 0
    rows.append({"offset_days": offset, "date": target, "status": status, "has_data": has_data})
    time.sleep(2)
pd.DataFrame(rows)

,offset_days,date,status,has_data
0,1,2026-08-09,200,True
1,3,2026-08-11,200,True
2,5,2026-08-13,200,True
3,7,2026-08-15,200,True
4,10,2026-08-18,200,True
5,14,2026-08-22,204,False
6,21,2026-08-29,204,False
7,30,2026-09-07,204,False
8,45,2026-09-22,204,False
9,60,2026-10-07,200,True
